# 🧠 Federated Active Causal Discovery with MARL (QMIX)
### Production Training & Evaluation Notebook (Kaggle GPU Edition)

This notebook trains decentralized agents to discover hidden causal graphs using JAX, Flax, and QMIX Value Factorization, with live experiment tracking on **Weights & Biases (WandB)**.

## Step 1: Install Dependencies & GPU Device Check

In [ ]:
!pip install -q wandb optax flax chex
import jax
print("JAX Devices:", jax.devices())
assert len(jax.devices()) > 0 and jax.devices()[0].platform in ['gpu', 'cuda', 'tpu', 'cpu']

## Step 2: Clone Repository

In [ ]:
import os
if not os.path.exists("federated-active-causal-discovery-with-marl"):
    !git clone https://github.com/BrianEzi/federated-active-causal-discovery-with-marl.git
%cd federated-active-causal-discovery-with-marl

## Step 3: Authenticate Weights & Biases (WandB)
*Tip: Add your `wandb_api_key` under Kaggle Add-ons -> Secrets to authenticate automatically.*

In [ ]:
import wandb
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("wandb_api_key")
    wandb.login(key=wandb_api_key)
    print("WandB authenticated via Kaggle Secret!")
except Exception as e:
    print("Kaggle Secret not found. Attempting interactive or disabled wandb session...")
    wandb.login(anonymous="allow")

## Step 4: Run Verification Test Suite

In [ ]:
!python -m pytest tests/ -v

## Step 5: Launch Industry-Standard Training Run
Customize your graph size, agent count, graph topology (ER/BA), and episode limits below.

In [ ]:
!python -m src.train \
    --num_variables 5 \
    --num_agents 2 \
    --graph_type ER \
    --edge_prob 0.4 \
    --num_episodes 100 \
    --batch_size 8 \
    --lr 0.001 \
    --eval_freq 10 \
    --use_wandb \
    --wandb_project "federated-causal-marl-kaggle"

## Step 6: Load & Inspect Best Model Checkpoint

In [ ]:
import numpy as np
checkpoint_path = "checkpoints/best_qmix_params.npz"
if os.path.exists(checkpoint_path):
    data = np.load(checkpoint_path, allow_pickle=True)
    print("Best QMIX Model Checkpoint successfully loaded!")
    print("Saved keys:", list(data.keys()))
else:
    print("No checkpoint saved yet.")